In [ ]:
# Setup & Dependencies
# --------------------
# It's recommended to run these in your terminal or a notebook cell
# !pip install -q -U transformers datasets accelerate peft bitsandbytes torch

import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
import gc
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import multiprocessing
from huggingface_hub import login

# Paste your token inside the quotes
hf_token = "" 

login(token=hf_token)

# --------------------
# Fix multiprocessing for CUDA
# --------------------
# Set the start method for multiprocessing to 'spawn' to avoid CUDA issues
if __name__ == '__main__':
    multiprocessing.set_start_method('spawn', force=True)

# --------------------
# Dataset Preparation - Updated to use separate paths
# --------------------

# Define separate paths for train, validation, and test sets
train_csv_path = "train.csv"
train_folder = "train"

val_csv_path = "val.csv"
val_folder = "val"

test_csv_path = "test.csv"
test_folder = "test"

save_directory = "llama_3.2_4"
os.makedirs(save_directory, exist_ok=True)

# Helper function to load and validate dataset
def load_and_validate_dataset(csv_path, image_folder, dataset_name="Dataset"):
    """Load CSV and validate image paths"""
    df = pd.read_csv(csv_path)
    extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
   
    valid_rows = []
    for _, row in df.iterrows():
        image_id = row['image_id']
        for ext in extensions:
            potential_path = os.path.join(image_folder, f"{image_id}{ext}")
            if os.path.exists(potential_path):
                row['filepath'] = potential_path
                valid_rows.append(row)
                break
   
    filtered_df = pd.DataFrame(valid_rows)
    print(f"{dataset_name}: Total valid images found: {len(filtered_df)}")
    return filtered_df

# Load all three datasets
train_df = load_and_validate_dataset(train_csv_path, train_folder, "Train")
val_df = load_and_validate_dataset(val_csv_path, val_folder, "Validation")
test_df = load_and_validate_dataset(test_csv_path, test_folder, "Test")

print(f"Final split - Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}\n")

# Fiber columns
fiber_columns = ['cotton', 'wool', 'polyester', 'linen', 'silk', 'other_fibres']

# --------------------------
# Device Setup
# --------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# --------------------------
# Dataset Class
# --------------------------

class FabricLlamaDataset(Dataset):
    def __init__(self, df):
        self.data = df.reset_index(drop=True)
        self.fibers = fiber_columns

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        try:
            image = Image.open(row['filepath']).convert("RGB")
            # Normalize fiber values to 0-1 range and handle NaN
            target = []
            for f in self.fibers:
                val = row[f]
                if pd.isna(val):
                    target.append(0.0)
                else:
                    # Always divide by 100 to convert percentage to 0-1 range
                    normalized_val = float(val) / 100.0
                    target.append(max(0.0, min(1.0, normalized_val)))

            target = torch.tensor(target, dtype=torch.float32)
            return {"image": image, "target": target}
        except Exception as e:
            print(f"Error loading image {row['filepath']}: {e}")
            dummy_image = Image.new('RGB', (224, 224), color='white')
            dummy_target = torch.zeros(len(self.fibers), dtype=torch.float32)
            return {"image": dummy_image, "target": dummy_target}
# --------------------------
# Processor and Collate Function
# --------------------------

model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
processor = AutoProcessor.from_pretrained(model_id)

def collate_fn(batch):
    images = [item["image"] for item in batch]
    targets = torch.stack([item["target"] for item in batch])
   
    messages = []
    for _ in range(len(images)):
        conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "What is the fabric composition of this textile? Please analyze the fibers present."}]}]
        messages.append(conversation)

    try:
        texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in messages]
        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
       
        # Keep inputs on CPU initially - they'll be moved to device in the model forward pass
        # Only move targets to device here
        targets = targets.to(device)
       
        return inputs, targets
       
    except Exception as e:
        print(f"Error in collate_fn: {e}")
        return None, None

# -----------------------------------
# LoRA Model Loading & Configuration
# -----------------------------------

# Clear GPU cache before loading
torch.cuda.empty_cache()
gc.collect()

# Fixed 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

# Custom device map to handle memory constraints
def create_device_map():
    """Create a custom device map for the model"""
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.get_device_properties(0).total_memory
        # If GPU has less than 16GB, use more aggressive CPU offloading
        if gpu_memory < 16 * 1024**3:  # Less than 16GB
            return {
                "language_model": 0,
                "vision_model": 0,
                "multi_modal_projector": 0,
                "lm_head": "cpu",
            }
        else:
            return "auto"
    else:
        return "cpu"

device_map = create_device_map()
print(f"Using device map: {device_map}")

print("Loading base Llama 3.2 Vision model with 4-bit quantization...")
try:
    llama_model = MllamaForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map=device_map,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    print("Base model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Trying with more conservative settings...")
   
    # Fallback configuration
    bnb_config_fallback = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True,
    )
   
    llama_model = MllamaForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config_fallback,
        device_map=device_map,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    print("Base model loaded with fallback 8-bit quantization.")

# Clear cache after loading
torch.cuda.empty_cache()
gc.collect()

# LoRA configuration - made more conservative
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

# Prepare model for k-bit training and apply LoRA
print("Preparing model for k-bit training...")
llama_model = prepare_model_for_kbit_training(llama_model)
llama_model_lora = get_peft_model(llama_model, lora_config)
print("LoRA adapters applied to the base model.")
llama_model_lora.print_trainable_parameters()

# Clear cache after LoRA application
torch.cuda.empty_cache()
gc.collect()

# ----------------------------------------------------
# Model with LoRA-adapted Backbone and Regression Head
# ----------------------------------------------------

class LlamaWithRegressor(nn.Module):
    def __init__(self, llama_peft_model, device):
        super().__init__()
        self.llama = llama_peft_model
        self.device = device
       
        # Get hidden size from the language model config
        if hasattr(self.llama.config, 'hidden_size'):
            hidden_size = self.llama.config.hidden_size
        elif hasattr(self.llama.config, 'language_model_config'):
            hidden_size = self.llama.config.language_model_config.hidden_size
        else:
            hidden_size = 4096
           
        self.model_dtype = next(self.llama.parameters()).dtype

        # Create a smaller, more efficient regression head
        self.regressor = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 6),
            nn.Sigmoid()
        ).to(dtype=self.model_dtype, device=self.device)

    def forward(self, **kwargs):
        # Move inputs to device explicitly
        device_kwargs = {}
        for key, value in kwargs.items():
            if isinstance(value, torch.Tensor):
                device_kwargs[key] = value.to(self.device)
            else:
                device_kwargs[key] = value
       
        # Handle mixed device placement
        with torch.amp.autocast(device_type='cuda', enabled=True, dtype=torch.bfloat16):
            outputs = self.llama(**device_kwargs, output_hidden_states=True, use_cache=False)
            hidden_states = outputs.hidden_states[-1]
           
            # Ensure hidden states are on the correct device
            if hidden_states.device != self.device:
                hidden_states = hidden_states.to(self.device)
           
            # Use the mean of the sequence for pooling
            pooled_output = hidden_states.mean(dim=1)
           
            # Pass through the regression head
            return self.regressor(pooled_output)

# Create the final model
print("Creating model with LoRA backbone and regression head...")
model = LlamaWithRegressor(llama_model_lora, device)
print("Model created successfully.")

# Clear cache after model creation
torch.cuda.empty_cache()
gc.collect()

# --------------------------
# Dataloaders
# --------------------------

BATCH_SIZE = 1

train_loader = DataLoader(
    FabricLlamaDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    FabricLlamaDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    FabricLlamaDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

# --------------------------
# Evaluation Functions
# --------------------------

def calculate_accuracy_metrics(predictions, targets, tolerance=0.1):
    predictions = predictions.cpu().numpy() * 100
    targets = targets.cpu().numpy() * 100
    mae = mean_absolute_error(targets.flatten(), predictions.flatten())
    mse = np.mean((predictions - targets) ** 2)
    rmse = np.sqrt(mse)
    r2 = r2_score(targets.flatten(), predictions.flatten())
    tolerance_accuracy = np.mean(np.abs(predictions - targets) <= (tolerance * 100)) * 100
    fiber_metrics = {f: {'MAE': mean_absolute_error(targets[:, i], predictions[:, i]), 'R²': r2_score(targets[:, i], predictions[:, i])} for i, f in enumerate(fiber_columns)}
    return {'MAE': mae, 'RMSE': rmse, 'R²': r2, 'Tolerance_Accuracy': tolerance_accuracy, 'Fiber_Metrics': fiber_metrics}

def evaluate_model(model, dataloader, loss_fn, device, dataset_name="Test"):
    model.eval()
    all_predictions, all_targets, losses = [], [], []
    print(f"\nEvaluating on {dataset_name} set...")
   
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(dataloader):
            if inputs is None:
                continue
           
            try:
                outputs = model(**inputs)
                outputs = torch.clamp(outputs.float(), 0.0, 1.0)
                loss = loss_fn(outputs, targets.float())
               
                if not (torch.isnan(loss) or torch.isinf(loss)):
                    losses.append(loss.item())
                    all_predictions.append(outputs)
                    all_targets.append(targets)
                   
            except Exception as e:
                print(f"Error in batch {batch_idx}: {e}")
                continue
           
            # Clear cache periodically
            if batch_idx % 10 == 0:
                torch.cuda.empty_cache()
               
    if not all_predictions:
        print(f"No valid predictions for {dataset_name} set!")
        return None
       
    all_predictions = torch.cat(all_predictions)
    all_targets = torch.cat(all_targets)
    avg_loss = np.mean(losses)
    metrics = calculate_accuracy_metrics(all_predictions, all_targets)
    return {'loss': avg_loss, 'metrics': metrics, 'predictions': all_predictions, 'targets': all_targets}

# --------------------------
# Training Loop
# --------------------------

loss_fn = nn.MSELoss()

# Get trainable parameters
trainable_params = list(model.llama.parameters()) + list(model.regressor.parameters())
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

scaler = torch.amp.GradScaler('cuda')

num_epochs = 60
best_val_loss = float('inf')
patience_counter = 0
max_patience = 8

print("Starting training...")

for epoch in range(num_epochs):
    model.train()
    train_losses, train_batches = [], 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        if inputs is None:
            continue
       
        try:
            optimizer.zero_grad()
           
            # Use mixed precision training with fixed autocast
            with torch.amp.autocast(device_type='cuda', enabled=True, dtype=torch.bfloat16):
                outputs = model(**inputs)
                outputs = torch.clamp(outputs.float(), 0.0, 1.0)
                loss = loss_fn(outputs, targets.float())

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"Skipping batch {batch_idx} due to invalid loss: {loss.item()}")
                continue
           
            # Scaled backward pass
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(loss.item())
            train_batches += 1
           
            if batch_idx % 10 == 0:
                print(f"Epoch {epoch+1}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.6f}")
               
            # More frequent cache clearing
            if batch_idx % 5 == 0:
                torch.cuda.empty_cache()
                gc.collect()
               
        except Exception as e:
            print(f"Error in training batch {batch_idx}: {e}")
            torch.cuda.empty_cache()
            gc.collect()
            continue

    # Validation phase
    torch.cuda.empty_cache()
    val_results = evaluate_model(model, val_loader, loss_fn, device, "Validation")
   
    if val_results:
        avg_train_loss = np.mean(train_losses) if train_losses else float('inf')
        avg_val_loss = val_results['loss']
        train_rmse_pct = (avg_train_loss ** 0.5) * 100
        val_rmse_pct = (avg_val_loss ** 0.5) * 100
       
        print(f"\nEpoch {epoch+1}/{num_epochs} Summary:")
        print(f"  Train Loss (RMSE): {train_rmse_pct:.2f}% | Raw: {avg_train_loss:.6f}")
        print(f"  Val Loss (RMSE):   {val_rmse_pct:.2f}% | Raw: {avg_val_loss:.6f}")
        print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
       
        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            print(f"  New best model found! Saving to {save_directory}")
           
            try:
                model.llama.save_pretrained(os.path.join(save_directory, 'best_lora_adapters'))
                torch.save(model.regressor.state_dict(), os.path.join(save_directory, 'best_regressor.pth'))
                processor.save_pretrained(os.path.join(save_directory, 'processor'))
            except Exception as e:
                print(f"Error saving model: {e}")
        else:
            patience_counter += 1
            if patience_counter >= max_patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break
    else:
        print(f"Epoch {epoch+1}/{num_epochs}: No valid validation batches processed.")

    print("-" * 60)
    torch.cuda.empty_cache()
    gc.collect()

# =========================
# FINAL TESTING PHASE
# =========================
from peft import PeftModel

print("\nTraining completed! Starting final evaluation...")

# Load the best model for testing
best_adapter_path = os.path.join(save_directory, 'best_lora_adapters')
best_regressor_path = os.path.join(save_directory, 'best_regressor.pth')

if os.path.exists(best_adapter_path) and os.path.exists(best_regressor_path):
    print("Loading best saved LoRA adapters and regressor head for testing...")
    try:
        # Clear current model from memory
        del model
        torch.cuda.empty_cache()
        gc.collect()
       
        # Load the base model again
        base_model = MllamaForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map=device_map,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )
       
        # Apply the saved LoRA adapters
        lora_model = PeftModel.from_pretrained(base_model, best_adapter_path)
       
        # Recreate the full model and load the regressor state
        test_model = LlamaWithRegressor(lora_model, device)
        test_model.regressor.load_state_dict(torch.load(best_regressor_path))
        print("Best model loaded successfully.")
       
    except Exception as e:
        print(f"Error loading best model: {e}")
        print("Using final model from training for testing.")
        test_model = model
else:
    print("Best model not found, using final model from training for testing.")
    test_model = model

# Helper functions for normalization
def softmax_numpy(x, temp=1.0):
    """Compute softmax values for each sets of scores in x."""
    x = x / temp
    e_x = np.exp(x - np.max(x, axis=0))
    return (e_x / e_x.sum(axis=0)) * 100

def top_k_normalization(x, k):
    """Zero out all but the top-k values and normalize them to sum to 100."""
    if len(x) <= k:
        x_sum = np.sum(x)
        return (x / x_sum) * 100 if x_sum > 0 else np.full_like(x, 100.0 / len(x))

    normalized_x = np.copy(x)
    non_top_k_indices = np.argsort(x)[:-k]
    normalized_x[non_top_k_indices] = 0
   
    top_k_sum = np.sum(normalized_x)
    if top_k_sum > 0:
        normalized_x = (normalized_x / top_k_sum) * 100
   
    return normalized_x

# Final testing using test set (only for evaluation)
torch.cuda.empty_cache()
test_results = evaluate_model(test_model, test_loader, loss_fn, device, "Test")

if test_results:
    print("\n" + "="*60)
    print("FINAL TEST RESULTS")
    print("="*60)

    metrics = test_results['metrics']
    test_loss_pct = (test_results['loss'] ** 0.5) * 100

    print(f"Test Loss (RMSE): {test_loss_pct:.2f}% | Raw: {test_results['loss']:.6f}")
    print(f"\nOverall Accuracy Metrics:")
    print(f"  Mean Absolute Error: {metrics['MAE']:.2f} pp")
    print(f"  Root Mean Square Error: {metrics['RMSE']:.2f} pp")
    print(f"  R² Score: {metrics['R²']:.4f}")
    print(f"  Tolerance Accuracy (±10%): {metrics['Tolerance_Accuracy']:.2f}%")

    print(f"\nPer-Fiber Performance:")
    print("-" * 40)
    for fiber, f_metrics in metrics['Fiber_Metrics'].items():
        print(f"{fiber.capitalize():12} | MAE: {f_metrics['MAE']:5.2f}% | R²: {f_metrics['R²']:6.4f}")

    # Sample predictions with normalization
    predictions = test_results['predictions'].cpu().numpy() * 100
    targets = test_results['targets'].cpu().numpy() * 100
   
    print(f"\n{'='*20} SAMPLE PREDICTIONS WITH NORMALIZATION (first 10 test samples) {'='*20}")
    print("Format: [Cotton, Wool, Polyester, Linen, Silk, Other]")
    print("-" * 75)
   
    for i in range(min(10, len(predictions))):
        pred_raw = predictions[i]
        targ = targets[i]
       
        pred_sum_raw_val = np.sum(pred_raw)
        pred_sum_norm = (pred_raw / pred_sum_raw_val) * 100 if pred_sum_raw_val > 0 else pred_raw
        pred_softmax_manual = softmax_numpy(pred_raw)
        pred_torch_tensor = torch.from_numpy(pred_raw)
        pred_softmax_torch = F.softmax(pred_torch_tensor, dim=0).numpy() * 100
        pred_top3_norm = top_k_normalization(pred_raw, k=3)
        pred_top4_norm = top_k_normalization(pred_raw, k=4)
       
        mae_raw = np.mean(np.abs(pred_raw - targ))
        mae_sum_norm = np.mean(np.abs(pred_sum_norm - targ))
        mae_softmax_manual = np.mean(np.abs(pred_softmax_manual - targ))
        mae_softmax_torch = np.mean(np.abs(pred_softmax_torch - targ))
        mae_top3_norm = np.mean(np.abs(pred_top3_norm - targ))
        mae_top4_norm = np.mean(np.abs(pred_top4_norm - targ))

        def format_array(arr):
            return ", ".join([f"{x:5.1f}" for x in arr])

        print(f"SAMPLE {i+1}")
        print(f"  ACTUAL:           [{format_array(targ)}] Sum: {np.sum(targ):5.1f}%")
        print("-" * 75)
        print(f"  Predicted (Raw):  [{format_array(pred_raw)}] Sum: {np.sum(pred_raw):5.1f}%\t| MAE: {mae_raw:.2f}%")
        print("\n--- Normalization Methods ---")
        print(f"  Sum Norm:         [{format_array(pred_sum_norm)}] Sum: {np.sum(pred_sum_norm):5.1f}%\t| MAE: {mae_sum_norm:.2f}%")
        print(f"  Softmax (NumPy):  [{format_array(pred_softmax_manual)}] Sum: {np.sum(pred_softmax_manual):5.1f}%\t| MAE: {mae_softmax_manual:.2f}%")
        print(f"  Softmax (Torch):  [{format_array(pred_softmax_torch)}] Sum: {np.sum(pred_softmax_torch):5.1f}%\t| MAE: {mae_softmax_torch:.2f}%")
        print(f"  Top-3 Norm:       [{format_array(pred_top3_norm)}] Sum: {np.sum(pred_top3_norm):5.1f}%\t| MAE: {mae_top3_norm:.2f}%")
        print(f"  Top-4 Norm:       [{format_array(pred_top4_norm)}] Sum: {np.sum(pred_top4_norm):5.1f}%\t| MAE: {mae_top4_norm:.2f}%")
        print("=" * 75 + "\n")

# Save final model components
print(f"\nSaving final model components to {save_directory}")
try:
    test_model.llama.save_pretrained(os.path.join(save_directory, 'final_lora_adapters'))
    torch.save(test_model.regressor.state_dict(), os.path.join(save_directory, 'final_regressor.pth'))
    processor.save_pretrained(os.path.join(save_directory, 'final_processor'))
    print("Model saved successfully!")
except Exception as e:
    print(f"Error saving final model: {e}")

print("\nTraining and testing completed!")

# Clear final cache
torch.cuda.empty_cache()
gc.collect()

/home/ccbd/Desktop/fabric/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: Total valid images found: 1584
Validation: Total valid images found: 198
Test: Total valid images found: 201
Final split - Train: 1584, Val: 198, Test: 201

Using device: cuda
GPU: NVIDIA GeForce RTX 4090
GPU Memory: 23.5 GB
Using device map: auto
Loading base Llama 3.2 Vision model with 4-bit quantization...


Loading checkpoint shards: 100%|██████████| 5/5 [00:15<00:00,  3.14s/it]


Base model loaded successfully.
Preparing model for k-bit training...
LoRA adapters applied to the base model.
trainable params: 11,796,480 || all params: 10,682,017,315 || trainable%: 0.1104
Creating model with LoRA backbone and regression head...
Model created successfully.
Starting training...


/home/ccbd/Desktop/fabric/.venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/ccbd/Desktop/fabric/.venv/lib/python3.10/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


...

/home/ccbd/Desktop/fabric/.venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/ccbd/Desktop/fabric/.venv/lib/python3.10/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch 7, Batch 0/1584, Loss: 0.162337
Epoch 7, Batch 10/1584, Loss: 0.000581
Epoch 7, Batch 20/1584, Loss: 0.017661
Epoch 7, Batch 30/1584, Loss: 0.059499
Epoch 7, Batch 40/1584, Loss: 0.002338
Epoch 7, Batch 50/1584, Loss: 0.054055
Epoch 7, Batch 60/1584, Loss: 0.141828
Epoch 7, Batch 70/1584, Loss: 0.002357
Epoch 7, Batch 80/1584, Loss: 0.021803
Epoch 7, Batch 90/1584, Loss: 0.008863
Epoch 7, Batch 100/1584, Loss: 0.031313
Epoch 7, Batch 110/1584, Loss: 0.019889
Epoch 7, Batch 120/1584, Loss: 0.007261
Epoch 7, Batch 130/1584, Loss: 0.001568
Epoch 7, Batch 140/1584, Loss: 0.156864
Epoch 7, Batch 150/1584, Loss: 0.002862
Epoch 7, Batch 160/1584, Loss: 0.109490
Epoch 7, Batch 170/1584, Loss: 0.019525
Epoch 7, Batch 180/1584, Loss: 0.202503
Epoch 7, Batch 190/1584, Loss: 0.001438
Epoch 7, Batch 200/1584, Loss: 0.004147
Epoch 7, Batch 210/1584, Loss: 0.200328
Epoch 7, Batch 220/1584, Loss: 0.003785
Epoch 7, Batch 230/1584, Loss: 0.008267
Epoch 7, Batch 240/1584, Loss: 0.003149
Epoch 7, Ba

/home/ccbd/Desktop/fabric/.venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/ccbd/Desktop/fabric/.venv/lib/python3.10/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch 8, Batch 0/1584, Loss: 0.039136
Epoch 8, Batch 10/1584, Loss: 0.014802
Epoch 8, Batch 20/1584, Loss: 0.132623
Epoch 8, Batch 30/1584, Loss: 0.002130
Epoch 8, Batch 40/1584, Loss: 0.183522
Epoch 8, Batch 50/1584, Loss: 0.125179
Epoch 8, Batch 60/1584, Loss: 0.000388
Epoch 8, Batch 70/1584, Loss: 0.002721
Epoch 8, Batch 80/1584, Loss: 0.000528
Epoch 8, Batch 90/1584, Loss: 0.000947


KeyboardInterrupt: 

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
import gc
from peft import PeftModel, prepare_model_for_kbit_training
from huggingface_hub import login

# Paste your token inside the quotes
hf_token = ""
login(token=hf_token)

# --------------------
# Configuration
# --------------------
model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
save_directory = "llama_3.2_4"
test_csv_path = "test.csv"
test_folder = "test"

# Fiber columns
fiber_columns = ['cotton', 'wool', 'polyester', 'linen', 'silk', 'other_fibres']

# --------------------------
# Device Setup
# --------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# --------------------------
# Dataset Class
# --------------------------
class FabricLlamaDataset(Dataset):
    def __init__(self, df):
        self.data = df.reset_index(drop=True)
        self.fibers = fiber_columns

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        try:
            image = Image.open(row['filepath']).convert("RGB")
            # Normalize fiber values to 0-1 range and handle NaN
            target = []
            for f in self.fibers:
                val = row[f]
                if pd.isna(val):
                    target.append(0.0)
                else:
                    # Always divide by 100 to convert percentage to 0-1 range
                    normalized_val = float(val) / 100.0
                    target.append(max(0.0, min(1.0, normalized_val)))

            target = torch.tensor(target, dtype=torch.float32)
            return {"image": image, "target": target}
        except Exception as e:
            print(f"Error loading image {row['filepath']}: {e}")
            dummy_image = Image.new('RGB', (224, 224), color='white')
            dummy_target = torch.zeros(len(self.fibers), dtype=torch.float32)
            return {"image": dummy_image, "target": dummy_target}

# --------------------------
# Processor and Collate Function
# --------------------------
processor = AutoProcessor.from_pretrained(model_id)

def collate_fn(batch):
    images = [item["image"] for item in batch]
    targets = torch.stack([item["target"] for item in batch])

    messages = []
    for _ in range(len(images)):
        conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "What is the fabric composition of this textile? Please analyze the fibers present."}]}]
        messages.append(conversation)

    try:
        texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in messages]
        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)

        targets = targets.to(device)

        return inputs, targets

    except Exception as e:
        print(f"Error in collate_fn: {e}")
        return None, None

# --------------------------
# Load and Validate Dataset
# --------------------------
def load_and_validate_dataset(csv_path, image_folder, dataset_name="Dataset"):
    """Load CSV and validate image paths"""
    df = pd.read_csv(csv_path)
    extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']

    valid_rows = []
    for _, row in df.iterrows():
        image_id = row['image_id']
        for ext in extensions:
            potential_path = os.path.join(image_folder, f"{image_id}{ext}")
            if os.path.exists(potential_path):
                row['filepath'] = potential_path
                valid_rows.append(row)
                break

    filtered_df = pd.DataFrame(valid_rows)
    print(f"{dataset_name}: Total valid images found: {len(filtered_df)}")
    return filtered_df

test_df = load_and_validate_dataset(test_csv_path, test_folder, "Test")
print(f"Test set size: {len(test_df)}\n")

# --------------------------
# Model Definition
# --------------------------
class LlamaWithRegressor(nn.Module):
    def __init__(self, llama_peft_model, device):
        super().__init__()
        self.llama = llama_peft_model
        self.device = device

        # Get hidden size from the language model config
        if hasattr(self.llama.config, 'hidden_size'):
            hidden_size = self.llama.config.hidden_size
        elif hasattr(self.llama.config, 'language_model_config'):
            hidden_size = self.llama.config.language_model_config.hidden_size
        else:
            hidden_size = 4096

        self.model_dtype = next(self.llama.parameters()).dtype

        # Create a smaller, more efficient regression head
        self.regressor = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 6),
            nn.Sigmoid()
        ).to(dtype=self.model_dtype, device=self.device)

    def forward(self, **kwargs):
        # Move inputs to device explicitly
        device_kwargs = {}
        for key, value in kwargs.items():
            if isinstance(value, torch.Tensor):
                device_kwargs[key] = value.to(self.device)
            else:
                device_kwargs[key] = value

        # Handle mixed device placement
        with torch.amp.autocast(device_type='cuda', enabled=True, dtype=torch.bfloat16):
            outputs = self.llama(**device_kwargs, output_hidden_states=True, use_cache=False)
            hidden_states = outputs.hidden_states[-1]

            # Ensure hidden states are on the correct device
            if hidden_states.device != self.device:
                hidden_states = hidden_states.to(self.device)

            # Use the mean of the sequence for pooling
            pooled_output = hidden_states.mean(dim=1)

            # Pass through the regression head
            return self.regressor(pooled_output)

# --------------------------
# Normalization Functions
# --------------------------
def softmax_numpy(x, temp=1.0):
    """Compute softmax values for each sets of scores in x."""
    x = x / temp
    e_x = np.exp(x - np.max(x, axis=0))
    return (e_x / e_x.sum(axis=0)) * 100

def top_k_normalization(x, k):
    """Zero out all but the top-k values and normalize them to sum to 100."""
    if len(x) <= k:
        x_sum = np.sum(x)
        return (x / x_sum) * 100 if x_sum > 0 else np.full_like(x, 100.0 / len(x))

    normalized_x = np.copy(x)
    non_top_k_indices = np.argsort(x)[:-k]
    normalized_x[non_top_k_indices] = 0

    top_k_sum = np.sum(normalized_x)
    if top_k_sum > 0:
        normalized_x = (normalized_x / top_k_sum) * 100

    return normalized_x

# --------------------------
# Evaluation Functions
# --------------------------
def calculate_accuracy_metrics(predictions, targets, tolerance=0.1):
    predictions = predictions.cpu().numpy() * 100
    targets = targets.cpu().numpy() * 100
    mae = mean_absolute_error(targets.flatten(), predictions.flatten())
    mse = np.mean((predictions - targets) ** 2)
    rmse = np.sqrt(mse)
    r2 = r2_score(targets.flatten(), predictions.flatten())
    tolerance_accuracy = np.mean(np.abs(predictions - targets) <= (tolerance * 100)) * 100
    fiber_metrics = {f: {'MAE': mean_absolute_error(targets[:, i], predictions[:, i]), 'R²': r2_score(targets[:, i], predictions[:, i])} for i, f in enumerate(fiber_columns)}
    return {'MAE': mae, 'RMSE': rmse, 'R²': r2, 'Tolerance_Accuracy': tolerance_accuracy, 'Fiber_Metrics': fiber_metrics}

def evaluate_model(model, dataloader, loss_fn, device, dataset_name="Test"):
    model.eval()
    all_predictions, all_targets, losses = [], [], []
    print(f"\nEvaluating on {dataset_name} set...")

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(dataloader):
            if inputs is None:
                continue

            try:
                outputs = model(**inputs)
                outputs = torch.clamp(outputs.float(), 0.0, 1.0)
                loss = loss_fn(outputs, targets.float())

                if not (torch.isnan(loss) or torch.isinf(loss)):
                    losses.append(loss.item())
                    all_predictions.append(outputs)
                    all_targets.append(targets)

            except Exception as e:
                print(f"Error in batch {batch_idx}: {e}")
                continue

            # Clear cache periodically
            if batch_idx % 10 == 0:
                torch.cuda.empty_cache()

    if not all_predictions:
        print(f"No valid predictions for {dataset_name} set!")
        return None

    all_predictions = torch.cat(all_predictions)
    all_targets = torch.cat(all_targets)
    avg_loss = np.mean(losses)
    metrics = calculate_accuracy_metrics(all_predictions, all_targets)
    return {'loss': avg_loss, 'metrics': metrics, 'predictions': all_predictions, 'targets': all_targets}

# --------------------------
# Load Best Model
# --------------------------
print("Loading best model from training...")

best_adapter_path = os.path.join(save_directory, 'best_lora_adapters')
best_regressor_path = os.path.join(save_directory, 'best_regressor.pth')

if os.path.exists(best_adapter_path) and os.path.exists(best_regressor_path):
    print("Found best model checkpoints. Loading...")
    try:
        torch.cuda.empty_cache()
        gc.collect()

        # 4-bit quantization configuration
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            llm_int8_enable_fp32_cpu_offload=True,
        )

        # Device map
        def create_device_map():
            if torch.cuda.is_available():
                gpu_memory = torch.cuda.get_device_properties(0).total_memory
                if gpu_memory < 16 * 1024**3:
                    return {
                        "language_model": 0,
                        "vision_model": 0,
                        "multi_modal_projector": 0,
                        "lm_head": "cpu",
                    }
                else:
                    return "auto"
            else:
                return "cpu"

        device_map = create_device_map()

        # Load base model
        print("Loading base model...")
        base_model = MllamaForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map=device_map,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )

        # Apply LoRA adapters
        print("Applying LoRA adapters...")
        lora_model = PeftModel.from_pretrained(base_model, best_adapter_path)

        # Create full model
        print("Creating model with regressor head...")
        test_model = LlamaWithRegressor(lora_model, device)
        test_model.regressor.load_state_dict(torch.load(best_regressor_path, map_location=device))
        print("Best model loaded successfully!\n")

    except Exception as e:
        print(f"Error loading best model: {e}")
        raise

else:
    print(f"Best model not found at {best_adapter_path} or {best_regressor_path}")
    raise FileNotFoundError("Best model checkpoints not found. Please train the model first.")

# --------------------------
# Create Test DataLoader
# --------------------------
test_loader = DataLoader(
    FabricLlamaDataset(test_df),
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

# --------------------------
# Run Testing
# --------------------------
print("Starting test evaluation...\n")

loss_fn = nn.MSELoss()
torch.cuda.empty_cache()
test_results = evaluate_model(test_model, test_loader, loss_fn, device, "Test")

if test_results:
    print("\n" + "="*60)
    print("FINAL TEST RESULTS")
    print("="*60)

    metrics = test_results['metrics']
    test_loss_pct = (test_results['loss'] ** 0.5) * 100

    print(f"Test Loss (RMSE): {test_loss_pct:.2f}% | Raw: {test_results['loss']:.6f}")
    print(f"\nOverall Accuracy Metrics:")
    print(f"  Mean Absolute Error: {metrics['MAE']:.2f} pp")
    print(f"  Root Mean Square Error: {metrics['RMSE']:.2f} pp")
    print(f"  R² Score: {metrics['R²']:.4f}")
    print(f"  Tolerance Accuracy (±10%): {metrics['Tolerance_Accuracy']:.2f}%")

    print(f"\nPer-Fiber Performance:")
    print("-" * 40)
    for fiber, f_metrics in metrics['Fiber_Metrics'].items():
        print(f"{fiber.capitalize():12} | MAE: {f_metrics['MAE']:5.2f}% | R²: {f_metrics['R²']:6.4f}")

    # Sample predictions with normalization
    predictions = test_results['predictions'].cpu().numpy() * 100
    targets = test_results['targets'].cpu().numpy() * 100

    print(f"\n{'='*20} SAMPLE PREDICTIONS WITH NORMALIZATION (first 10 test samples) {'='*20}")
    print("Format: [Cotton, Wool, Polyester, Linen, Silk, Other]")
    print("-" * 75)

    for i in range(min(10, len(predictions))):
        pred_raw = predictions[i]
        targ = targets[i]

        pred_sum_raw_val = np.sum(pred_raw)
        pred_sum_norm = (pred_raw / pred_sum_raw_val) * 100 if pred_sum_raw_val > 0 else pred_raw
        pred_softmax_manual = softmax_numpy(pred_raw)
        pred_torch_tensor = torch.from_numpy(pred_raw)
        pred_softmax_torch = F.softmax(pred_torch_tensor, dim=0).numpy() * 100
        pred_top3_norm = top_k_normalization(pred_raw, k=3)
        pred_top4_norm = top_k_normalization(pred_raw, k=4)

        mae_raw = np.mean(np.abs(pred_raw - targ))
        mae_sum_norm = np.mean(np.abs(pred_sum_norm - targ))
        mae_softmax_manual = np.mean(np.abs(pred_softmax_manual - targ))
        mae_softmax_torch = np.mean(np.abs(pred_softmax_torch - targ))
        mae_top3_norm = np.mean(np.abs(pred_top3_norm - targ))
        mae_top4_norm = np.mean(np.abs(pred_top4_norm - targ))

        def format_array(arr):
            return ", ".join([f"{x:5.1f}" for x in arr])

        print(f"SAMPLE {i+1}")
        print(f"  ACTUAL:           [{format_array(targ)}] Sum: {np.sum(targ):5.1f}%")
        print("-" * 75)
        print(f"  Predicted (Raw):  [{format_array(pred_raw)}] Sum: {np.sum(pred_raw):5.1f}%\t| MAE: {mae_raw:.2f}%")
        print("\n--- Normalization Methods ---")
        print(f"  Sum Norm:         [{format_array(pred_sum_norm)}] Sum: {np.sum(pred_sum_norm):5.1f}%\t| MAE: {mae_sum_norm:.2f}%")
        print(f"  Softmax (NumPy):  [{format_array(pred_softmax_manual)}] Sum: {np.sum(pred_softmax_manual):5.1f}%\t| MAE: {mae_softmax_manual:.2f}%")
        print(f"  Softmax (Torch):  [{format_array(pred_softmax_torch)}] Sum: {np.sum(pred_softmax_torch):5.1f}%\t| MAE: {mae_softmax_torch:.2f}%")
        print(f"  Top-3 Norm:       [{format_array(pred_top3_norm)}] Sum: {np.sum(pred_top3_norm):5.1f}%\t| MAE: {mae_top3_norm:.2f}%")
        print(f"  Top-4 Norm:       [{format_array(pred_top4_norm)}] Sum: {np.sum(pred_top4_norm):5.1f}%\t| MAE: {mae_top4_norm:.2f}%")
        print("=" * 75 + "\n")

    print("Testing completed successfully!")

else:
    print("No valid test results obtained.")

torch.cuda.empty_cache()
gc.collect()

Using device: cuda
GPU: NVIDIA GeForce RTX 4090
GPU Memory: 23.5 GB
Test: Total valid images found: 201
Test set size: 201

Loading best model from training...
Found best model checkpoints. Loading...
Loading base model...


Loading checkpoint shards: 100%|██████████| 5/5 [00:08<00:00,  1.78s/it]


Applying LoRA adapters...
Creating model with regressor head...
Best model loaded successfully!

Starting test evaluation...


Evaluating on Test set...


/tmp/ipykernel_64409/3722433162.py:313: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_model.regressor.load_state_dict(torch.load(best_regressor_path, map_location=devic


FINAL TEST RESULTS
Test Loss (RMSE): 25.37% | Raw: 0.064345

Overall Accuracy Metrics:
  Mean Absolute Error: 14.87 pp
  Root Mean Square Error: 25.37 pp
  R² Score: 0.4165
  Tolerance Accuracy (±10%): 63.60%

Per-Fiber Performance:
----------------------------------------
Cotton       | MAE: 30.36% | R²: 0.2392
Wool         | MAE: 14.15% | R²: 0.4502
Polyester    | MAE: 24.14% | R²: 0.1871
Linen        | MAE: 12.17% | R²: 0.4063
Silk         | MAE:  6.43% | R²: 0.0951
Other_fibres | MAE:  1.98% | R²: -0.0216

==================== SAMPLE PREDICTIONS WITH NORMALIZATION (first 10 test samples) ====================
Format: [Cotton, Wool, Polyester, Linen, Silk, Other]
---------------------------------------------------------------------------
SAMPLE 1
  ACTUAL:           [ 99.0,   0.0,   0.0,   0.0,   0.0,   1.0] Sum: 100.0%
---------------------------------------------------------------------------
  Predicted (Raw):  [ 60.2,   2.8,  11.4,   3.7,   1.2,   0.9] Sum:  80.2%	| MAE: 9.70%



64

: 